# Detailed Disease Biology Clustering Analysis
## Exploring Disease Modules and Morphological Patterns

This notebook provides detailed visualizations and analyses of disease biology patterns within variant clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys
sys.path.append('../..')
from analyses_plot_utils import *

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Load Data

In [ ]:
# Load all generated data
clustering_df = pd.read_csv('../../3_outputs/clustering.csv', index_col=0)
disease_modules_df = pd.read_csv('../../3_outputs/clustering_with_disease_modules.csv')
cluster_summary_df = pd.read_csv('../../3_outputs/cluster_disease_summary.csv')
interesting_clusters_df = pd.read_csv('../../3_outputs/interesting_disease_clusters.csv')

print(f"Total variants: {len(disease_modules_df)}")
print(f"Variants with disease info: {disease_modules_df['disease_count'].gt(0).sum()}")
print(f"Total clusters: {disease_modules_df['cluster'].nunique()}")
print(f"\nInteresting clusters identified: {len(interesting_clusters_df)}")

## 2. Cluster Overview and Composition

In [ ]:
# Display cluster summary
print("\n" + "="*80)
print("CLUSTER SUMMARY")
print("="*80)
display(cluster_summary_df)

In [ ]:
# Visualize cluster sizes and disease module distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cluster sizes
ax = axes[0, 0]
cluster_summary_df.plot(x='cluster', y='total_variants', kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of Variants')
ax.set_title('Cluster Sizes', fontsize=14, fontweight='bold')
ax.legend().remove()

# 2. Mean AUROC by cluster
ax = axes[0, 1]
cluster_summary_df.plot(x='cluster', y='mean_AUROC', kind='bar', ax=ax, color='coral')
ax.set_xlabel('Cluster')
ax.set_ylabel('Mean AUROC')
ax.set_title('Mean AUROC by Cluster', fontsize=14, fontweight='bold')
ax.axhline(y=0.95, color='red', linestyle='--', label='AUROC=0.95')
ax.legend()
ax.set_ylim([0.93, 0.99])

# 3. Altered variants percentage
ax = axes[1, 0]
cluster_summary_df['altered_pct'] = (cluster_summary_df['altered_variants'] / 
                                      cluster_summary_df['total_variants'] * 100)
cluster_summary_df.plot(x='cluster', y='altered_pct', kind='bar', ax=ax, color='darkgreen')
ax.set_xlabel('Cluster')
ax.set_ylabel('% Altered Variants')
ax.set_title('Percentage of Altered Variants by Cluster', fontsize=14, fontweight='bold')
ax.axhline(y=80, color='red', linestyle='--', label='80%')
ax.legend()

# 4. Module diversity
ax = axes[1, 1]
cluster_summary_df.plot(x='cluster', y='module_diversity', kind='bar', ax=ax, color='purple')
ax.set_xlabel('Cluster')
ax.set_ylabel('Number of Disease Modules')
ax.set_title('Disease Module Diversity by Cluster', fontsize=14, fontweight='bold')
ax.legend().remove()

plt.tight_layout()
plt.savefig('../../3_outputs/cluster_overview_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: cluster_overview_metrics.png")

## 3. Disease Module Distribution Across Clusters

In [ ]:
# Create disease module composition heatmap
module_cluster_counts = pd.crosstab(disease_modules_df['primary_disease_module'], 
                                     disease_modules_df['cluster'])

# Calculate percentages within each cluster
module_cluster_pct = module_cluster_counts.div(module_cluster_counts.sum(axis=0), axis=1) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Absolute counts
ax = axes[0]
sns.heatmap(module_cluster_counts, annot=True, fmt='d', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Disease Module', fontsize=12)
ax.set_title('Disease Module Distribution Across Clusters (Counts)', fontsize=14, fontweight='bold')

# Percentages
ax = axes[1]
sns.heatmap(module_cluster_pct, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, cbar_kws={'label': '% of Cluster'})
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Disease Module', fontsize=12)
ax.set_title('Disease Module Enrichment (% within Cluster)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../../3_outputs/disease_module_heatmaps.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: disease_module_heatmaps.png")

In [ ]:
# Stacked bar chart of disease module composition
fig, ax = plt.subplots(figsize=(14, 8))

module_cluster_pct.T.plot(kind='bar', stacked=True, ax=ax, 
                          colormap='tab10', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Percentage of Variants', fontsize=12)
ax.set_title('Disease Module Composition by Cluster', fontsize=14, fontweight='bold')
ax.legend(title='Disease Module', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('../../3_outputs/disease_module_stacked_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: disease_module_stacked_bar.png")

## 4. Gene-Specific Clustering Patterns

In [ ]:
# Extract gene from gene_allele
disease_modules_df['gene'] = disease_modules_df['gene_allele'].str.split('_').str[0]

# Find genes with multiple variants
gene_counts = disease_modules_df['gene'].value_counts()
multi_variant_genes = gene_counts[gene_counts >= 3].index.tolist()

print(f"\nGenes with 3+ variants: {len(multi_variant_genes)}")
print("\nTop 15 genes by variant count:")
print(gene_counts.head(15))

In [ ]:
# Analyze clustering consistency for top genes
top_genes = gene_counts.head(10).index.tolist()

gene_cluster_analysis = []
for gene in top_genes:
    gene_data = disease_modules_df[disease_modules_df['gene'] == gene]
    clusters = gene_data['cluster'].value_counts()
    primary_cluster = clusters.index[0]
    primary_count = clusters.iloc[0]
    total_count = len(gene_data)
    clustering_consistency = (primary_count / total_count * 100)
    
    gene_cluster_analysis.append({
        'gene': gene,
        'total_variants': total_count,
        'primary_cluster': primary_cluster,
        'variants_in_primary': primary_count,
        'clustering_consistency_%': clustering_consistency,
        'num_clusters': len(clusters),
        'mean_AUROC': gene_data['AUROC'].mean(),
        'primary_disease_module': gene_data['primary_disease_module'].mode()[0] if len(gene_data) > 0 else 'N/A'
    })

gene_cluster_df = pd.DataFrame(gene_cluster_analysis)
gene_cluster_df = gene_cluster_df.sort_values('clustering_consistency_%', ascending=False)

print("\n" + "="*80)
print("GENE CLUSTERING CONSISTENCY ANALYSIS")
print("="*80)
display(gene_cluster_df)

In [ ]:
# Visualize gene clustering consistency
fig, ax = plt.subplots(figsize=(12, 8))

colors = plt.cm.RdYlGn(gene_cluster_df['clustering_consistency_%'] / 100)
bars = ax.barh(gene_cluster_df['gene'], gene_cluster_df['clustering_consistency_%'], color=colors)
ax.set_xlabel('Clustering Consistency (%)', fontsize=12)
ax.set_ylabel('Gene', fontsize=12)
ax.set_title('Gene Clustering Consistency\n(% of variants in primary cluster)', 
             fontsize=14, fontweight='bold')
ax.axvline(x=80, color='red', linestyle='--', label='80% threshold')
ax.legend()
ax.set_xlim([0, 105])

# Add variant counts as text
for i, (idx, row) in enumerate(gene_cluster_df.iterrows()):
    ax.text(row['clustering_consistency_%'] + 2, i, 
            f"n={row['total_variants']}", va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../3_outputs/gene_clustering_consistency.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: gene_clustering_consistency.png")

## 5. Detailed Analysis of Interesting Clusters

In [ ]:
# Display interesting clusters
print("\n" + "="*80)
print("INTERESTING DISEASE BIOLOGY CLUSTERS")
print("="*80)
display(interesting_clusters_df)

In [ ]:
# Detailed analysis of each interesting cluster
for idx, cluster_info in interesting_clusters_df.iterrows():
    cluster_id = cluster_info['cluster']
    cluster_data = disease_modules_df[disease_modules_df['cluster'] == cluster_id]
    
    print("\n" + "="*80)
    print(f"CLUSTER {cluster_id}: {cluster_info['disease_module']} Enrichment")
    print("="*80)
    print(f"Enrichment: {cluster_info['enrichment_pct']:.1f}%")
    print(f"Variant count: {cluster_info['variant_count']}")
    print(f"Mean AUROC: {cluster_info['mean_AUROC']:.3f}")
    print(f"\nKey diseases: {cluster_info['key_diseases']}")
    
    # Show gene distribution
    print(f"\nGene distribution:")
    gene_dist = cluster_data['gene'].value_counts().head(10)
    for gene, count in gene_dist.items():
        print(f"  {gene}: {count} variants")
    
    # Show AUROC distribution
    print(f"\nAUROC statistics:")
    print(f"  Min: {cluster_data['AUROC'].min():.3f}")
    print(f"  Max: {cluster_data['AUROC'].max():.3f}")
    print(f"  Mean: {cluster_data['AUROC'].mean():.3f}")
    print(f"  Median: {cluster_data['AUROC'].median():.3f}")

In [ ]:
# Create detailed visualizations for interesting clusters
n_interesting = len(interesting_clusters_df)
fig, axes = plt.subplots(n_interesting, 3, figsize=(18, 6*n_interesting))

if n_interesting == 1:
    axes = axes.reshape(1, -1)

for i, (idx, cluster_info) in enumerate(interesting_clusters_df.iterrows()):
    cluster_id = cluster_info['cluster']
    cluster_data = disease_modules_df[disease_modules_df['cluster'] == cluster_id]
    
    # Gene distribution
    ax = axes[i, 0]
    gene_counts_cluster = cluster_data['gene'].value_counts().head(10)
    gene_counts_cluster.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Number of Variants')
    ax.set_title(f'Cluster {cluster_id}: Gene Distribution', fontweight='bold')
    
    # AUROC distribution
    ax = axes[i, 1]
    ax.hist(cluster_data['AUROC'], bins=20, color='coral', edgecolor='black', alpha=0.7)
    ax.axvline(cluster_data['AUROC'].mean(), color='red', linestyle='--', 
               label=f'Mean: {cluster_data["AUROC"].mean():.3f}')
    ax.set_xlabel('AUROC')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Cluster {cluster_id}: AUROC Distribution', fontweight='bold')
    ax.legend()
    
    # Disease module composition
    ax = axes[i, 2]
    module_counts = cluster_data['primary_disease_module'].value_counts()
    colors_pie = plt.cm.Set3(range(len(module_counts)))
    ax.pie(module_counts.values, labels=module_counts.index, autopct='%1.1f%%',
           colors=colors_pie, startangle=90)
    ax.set_title(f'Cluster {cluster_id}: Disease Module Composition', fontweight='bold')

plt.tight_layout()
plt.savefig('../../3_outputs/interesting_clusters_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: interesting_clusters_detailed.png")

## 6. UMAP Visualization with Disease Module Annotations

In [ ]:
# Merge UMAP coordinates with disease modules
umap_data = clustering_df[['UMAP1', 'UMAP2', 'Metadata_cluster']].copy()
umap_data['gene_allele'] = umap_data.index
umap_merged = umap_data.merge(disease_modules_df[['gene_allele', 'primary_disease_module', 'AUROC', 'gene']], 
                               on='gene_allele', how='left')

print(f"UMAP data points: {len(umap_merged)}")

In [ ]:
# Create UMAP visualizations
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# By cluster
ax = axes[0]
for cluster in sorted(umap_merged['Metadata_cluster'].unique()):
    cluster_points = umap_merged[umap_merged['Metadata_cluster'] == cluster]
    ax.scatter(cluster_points['UMAP1'], cluster_points['UMAP2'], 
               label=f'Cluster {cluster}', s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
ax.set_xlabel('UMAP1', fontsize=12)
ax.set_ylabel('UMAP2', fontsize=12)
ax.set_title('UMAP: Colored by Cluster', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

# By disease module
ax = axes[1]
disease_modules = umap_merged['primary_disease_module'].dropna().unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(disease_modules)))
for i, module in enumerate(sorted(disease_modules)):
    module_points = umap_merged[umap_merged['primary_disease_module'] == module]
    ax.scatter(module_points['UMAP1'], module_points['UMAP2'], 
               label=module, s=80, alpha=0.7, color=colors[i], edgecolors='black', linewidth=0.5)
ax.set_xlabel('UMAP1', fontsize=12)
ax.set_ylabel('UMAP2', fontsize=12)
ax.set_title('UMAP: Colored by Disease Module', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

# By AUROC
ax = axes[2]
scatter = ax.scatter(umap_merged['UMAP1'], umap_merged['UMAP2'], 
                     c=umap_merged['AUROC'], cmap='RdYlGn', s=80, 
                     alpha=0.7, edgecolors='black', linewidth=0.5, vmin=0.93, vmax=1.0)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('AUROC', fontsize=12)
ax.set_xlabel('UMAP1', fontsize=12)
ax.set_ylabel('UMAP2', fontsize=12)
ax.set_title('UMAP: Colored by AUROC', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../3_outputs/umap_disease_module_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: umap_disease_module_visualization.png")

## 7. Top Genes Highlighted on UMAP

In [ ]:
# Highlight top genes on UMAP
top_genes_to_plot = ['GFAP', 'MLH1', 'PLP1', 'LITAF', 'RAD51D', 'BRCA1']

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

for i, gene in enumerate(top_genes_to_plot):
    ax = axes[i]
    
    # Plot all points in gray
    ax.scatter(umap_merged['UMAP1'], umap_merged['UMAP2'], 
               c='lightgray', s=50, alpha=0.3, edgecolors='none')
    
    # Highlight specific gene
    gene_points = umap_merged[umap_merged['gene'] == gene]
    if len(gene_points) > 0:
        scatter = ax.scatter(gene_points['UMAP1'], gene_points['UMAP2'], 
                           c=gene_points['AUROC'], cmap='RdYlGn', s=150, 
                           alpha=0.9, edgecolors='black', linewidth=1.5, vmin=0.93, vmax=1.0)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('AUROC', fontsize=10)
        
        # Get cluster info
        clusters = gene_points['Metadata_cluster'].value_counts()
        primary_cluster = clusters.index[0]
        consistency = clusters.iloc[0] / len(gene_points) * 100
        
        ax.set_title(f'{gene} variants (n={len(gene_points)})\n' + 
                    f'Primary cluster: {primary_cluster} ({consistency:.0f}%)',
                    fontsize=12, fontweight='bold')
    else:
        ax.set_title(f'{gene}: No variants found', fontsize=12)
    
    ax.set_xlabel('UMAP1', fontsize=10)
    ax.set_ylabel('UMAP2', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../3_outputs/umap_top_genes_highlighted.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: umap_top_genes_highlighted.png")

## 8. Cross-Disease Module Analysis

In [ ]:
# Analyze variants that bridge multiple disease modules within a cluster
print("\n" + "="*80)
print("CROSS-DISEASE MODULE ANALYSIS")
print("="*80)

for cluster_id in sorted(disease_modules_df['cluster'].unique()):
    cluster_data = disease_modules_df[disease_modules_df['cluster'] == cluster_id]
    modules_in_cluster = cluster_data['primary_disease_module'].unique()
    
    if len(modules_in_cluster) > 1:
        print(f"\nCluster {cluster_id}:")
        print(f"  Contains {len(modules_in_cluster)} disease modules:")
        for module in modules_in_cluster:
            module_data = cluster_data[cluster_data['primary_disease_module'] == module]
            print(f"    - {module}: {len(module_data)} variants (mean AUROC: {module_data['AUROC'].mean():.3f})")
        
        # Check if there are genes spanning multiple modules
        gene_module_pairs = cluster_data.groupby('gene')['primary_disease_module'].nunique()
        multi_module_genes = gene_module_pairs[gene_module_pairs > 1]
        if len(multi_module_genes) > 0:
            print(f"  Genes with variants in multiple modules: {list(multi_module_genes.index)}")

## 9. Summary Statistics and Key Insights

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS AND SUMMARY STATISTICS")
print("="*80)

print("\n1. OVERALL STATISTICS:")
print(f"   - Total variants analyzed: {len(disease_modules_df)}")
print(f"   - Variants with disease annotations: {disease_modules_df['disease_count'].gt(0).sum()}")
print(f"   - Total clusters: {disease_modules_df['cluster'].nunique()}")
print(f"   - Disease modules identified: {disease_modules_df['primary_disease_module'].nunique()}")
print(f"   - Mean AUROC across all variants: {disease_modules_df['AUROC'].mean():.3f}")

print("\n2. CLUSTERING CONSISTENCY:")
highly_consistent_genes = gene_cluster_df[gene_cluster_df['clustering_consistency_%'] >= 80]
print(f"   - Genes with >=80% clustering consistency: {len(highly_consistent_genes)}")
print(f"   - Most consistent gene: {gene_cluster_df.iloc[0]['gene']} ({gene_cluster_df.iloc[0]['clustering_consistency_%']:.1f}%)")

print("\n3. DISEASE MODULE ENRICHMENT:")
print(f"   - Clusters with >50% single-module enrichment: {len(interesting_clusters_df)}")
max_enrichment = cluster_summary_df['dominant_module_pct'].max()
max_enrichment_cluster = cluster_summary_df.loc[cluster_summary_df['dominant_module_pct'].idxmax()]
print(f"   - Highest enrichment: Cluster {max_enrichment_cluster['cluster']} ({max_enrichment:.1f}%)")
print(f"   - Module: {max_enrichment_cluster['dominant_disease_module']}")

print("\n4. BIOLOGICAL IMPLICATIONS:")
print("   - Gene-specific morphological signatures strongly supported")
print("   - Disease modules show distinct but overlapping cellular phenotypes")
print("   - High AUROC variants tend to cluster together")
print("   - Some disease modules span multiple morphological clusters")

print("\n5. TOP FINDINGS:")
for i, row in interesting_clusters_df.iterrows():
    print(f"   - Cluster {row['cluster']}: {row['enrichment_pct']:.0f}% {row['disease_module']} enrichment")
    print(f"     ({row['variant_count']} variants, mean AUROC: {row['mean_AUROC']:.3f})")

## 10. Export Summary Tables

In [ ]:
# Export gene clustering consistency table
gene_cluster_df.to_csv('../../3_outputs/gene_clustering_consistency.csv', index=False)
print("Saved: gene_clustering_consistency.csv")

# Export UMAP data with annotations
umap_merged.to_csv('../../3_outputs/umap_with_disease_annotations.csv', index=False)
print("Saved: umap_with_disease_annotations.csv")

print("\nAll analyses complete!")